# Day 35 · 端到端联调

**配套讲义**: [`days/day-35.md`](../days/day-35.md) ｜ **需要 GPU（云机器）**

做一个 Gradio 界面：能上传图 + 打字，**能看到 Agent 的思考过程和工具调用**；在 20 条真实场景 query 上跑通，人工判定成功率。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 起界面（在终端里跑）

In [ ]:
print("""
在终端执行（不要在这个 notebook 里跑，Gradio 会阻塞）：

    python -m src.agent.demo --port 7860

手机上看：

    python -m src.agent.demo --share
""")

## 2. 准备 20 条真实场景 query（今天的主产出）

从你 Day 8 抄来的真实问法里挑 20 条，覆盖 8 个意图。

In [ ]:
real_queries = [
    ("这件米白针织衫有货吗？", None),                 # (问题, 图片路径)
    ("订单 A1 到哪了？", None),
    ("我要退货，尺码不合适", None),
    # ... 补齐到 20 条，覆盖 8 个意图
]
from collections import Counter
print(f"共 {len(real_queries)} 条，还差 {max(0, 20 - len(real_queries))} 条")

## 3. 人工判定表

In [ ]:
import json
from pathlib import Path

verdicts = {"成功": [], "部分成功": [], "失败": []}
RESULT_FILE = "../reports/agent_demo_20.jsonl"
if Path(RESULT_FILE).exists():
    rows = [json.loads(l) for l in Path(RESULT_FILE).read_text().splitlines() if l.strip()]
    for r in rows:
        verdicts[r.get("verdict", "失败")].append(r.get("query"))
    for k, v in verdicts.items():
        print(f"{k}: {len(v)} 条  ({len(v)/max(len(rows),1):.0%})")
    print("\n失败集中在:", verdicts["失败"][:3])
else:
    print("先在界面里跑完 20 条，把结果记到 reports/agent_demo_20.jsonl")

## 验收清单

- [ ] 界面能上传图 + 打字，且**能看到工具调用过程**
- [ ] 20 条真实场景 query 跑通，人工判定成功率（≥65% 为 M6 目标）
- [ ] 至少能录一段 60 秒的 demo 视频（这是 W8 交付物的素材）
- [ ] 错误路径也好看（工具失败时显示友好提示，不是红字 traceback）

**卡住了？** 回看 [`days/day-35.md`](../days/day-35.md) 第五节「容易踩的坑」。

> **明天**：`days/day-36.md` —— Agent 评测，W6 收官